In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lynnxxx/spotify-tracks-popularity-classification")

print("Path to dataset files:", path)

import os
import pandas as pd

# The 'path' variable already holds the directory for the new dataset

# 1. List files in the directory to find the main CSV file
csv_files_new_dataset = [f for f in os.listdir(path) if f.endswith('.csv')]

if not csv_files_new_dataset:
    print(f"No CSV files found in the directory: {path}")
else:
    # Assuming the main data file is the first CSV found or has a specific name
    # Let's try to find a file that seems relevant, otherwise take the first one.
    # Common names for such files are 'tracks.csv', 'spotify_tracks.csv', etc.
    # For now, let's assume it's the first one.
    audio_popularity_file_name = csv_files_new_dataset[0]
    audio_popularity_file_path = os.path.join(path, audio_popularity_file_name)

    # 2. Load the identified CSV file into a new DataFrame
    df_audio_popularity = pd.read_csv(audio_popularity_file_path)

    print(f"Loaded file: {audio_popularity_file_path}")
    print("First 5 rows of the audio popularity dataset:")
    print(df_audio_popularity.head())
    print(f"\nShape of df_audio_popularity: {df_audio_popularity.shape}")

import numpy as np

# 1. Create a common identifier in df_lyrics for merging
# The 'song_id' in df_lyrics is 'spotify:track:ID', and 'id' in df_audio_popularity is 'ID'.
df_lyrics['track_id_for_merge'] = df_lyrics['song_id'].apply(lambda x: x.split(':')[-1] if isinstance(x, str) else np.nan)

# 2. Select the relevant columns (track ID and 'popularity') from df_audio_popularity
df_popularity_info = df_audio_popularity[['id', 'popularity']]

# 3. Merge these selected columns with the existing df_lyrics DataFrame
df_lyrics_merged = pd.merge(
    df_lyrics,
    df_popularity_info,
    left_on='track_id_for_merge',
    right_on='id',
    how='left'
)

# Drop the redundant 'id' and 'track_id_for_merge' columns after merge
df_lyrics_merged = df_lyrics_merged.drop(columns=['id', 'track_id_for_merge'])

# 4. Rename the newly merged 'popularity' column to 'popularity_from_audio'
df_lyrics_merged = df_lyrics_merged.rename(columns={'popularity': 'popularity_from_audio'})

# 5. Handle any missing values in the new 'popularity_from_audio' column
# For simplicity, let's fill NaN values with 0, assuming missing popularity means not popular.
df_lyrics_merged['popularity_from_audio'] = df_lyrics_merged['popularity_from_audio'].fillna(0)

# 6. Create a new binary target variable 'is_hit_popularity'
# This variable should be 1 if 'popularity_from_audio' is >= 50, and 0 otherwise.
df_lyrics_merged['is_hit_popularity'] = (df_lyrics_merged['popularity_from_audio'] >= 50).astype(int)

# Update df_lyrics to the merged DataFrame
df_lyrics = df_lyrics_merged

print("First 5 rows of df_lyrics with merged popularity and new target variable:")
print(df_lyrics[['song_id', 'popularity_from_audio', 'is_hit_popularity']].head())
print(f"\nShape of df_lyrics: {df_lyrics.shape}")
print(f"Value counts for is_hit_popularity:\n{df_lyrics['is_hit_popularity'].value_counts()}")

## Train and Evaluate Gaussian Naive Bayes (Audio Features for Popularity)

### Subtask:
Prepare the audio features data from `df_audio_popularity`, define the 'is_hit' target, and then train and evaluate a Gaussian Naive Bayes model.


**Reasoning**:
First, I will define the features (X) by selecting the specified audio-related columns from `df_audio_popularity` and define the target variable (y) as `is_hit` based on the 'popularity' column from the same DataFrame. This step prepares the data for model training.



import pandas as pd
import numpy as np

# 1. Define the features (X) for the model
# Select audio-related columns from df_audio_popularity
# Ensure 'popularity' column name is correctly handled as per previous step where it was renamed to 'popularity_from_audio'
# Re-loading df_audio_popularity from kernel state, as df_lyrics was updated with merged data.
# The task specifically mentions 'df_audio_popularity' so I will re-extract from that specific df.

# Let's get the original df_audio_popularity from the kernel state as `df_lyrics` was modified.
df_audio_popularity = pd.read_csv(audio_popularity_file_path)

# Columns to use as features
features_audio = ['acousticness', 'danceability', 'energy', 'valence', 'tempo', 'loudness', 'speechiness', 'instrumentalness']
X_audio = df_audio_popularity[features_audio]

# 2. Define the target variable (y) as a binary 'is_hit'
# A song is a 'hit' if its popularity (from df_audio_popularity) is 50 or greater.
# Ensure the column name 'popularity' is correct in df_audio_popularity
y_audio = (df_audio_popularity['popularity'] >= 50).astype(int)

print("First 5 rows of X_audio (features):")
print(X_audio.head())
print("\nValue counts for y_audio (target 'is_hit'):")
print(y_audio.value_counts())

**Reasoning**:
Now that the features (`X_audio`) and target variable (`y_audio`) are defined, the next step is to split them into training and testing sets to prepare for model training and evaluation. I will use `train_test_split` with `stratify=y_audio` to ensure an even distribution of 'hit' and 'non-hit' songs in both sets.



from sklearn.model_selection import train_test_split

# 3. Split the features (X) and target (y) into training and testing sets
# Ensuring a stratified split to maintain the proportion of 'hit' and 'non-hit' songs
X_train_audio, X_test_audio, y_train_audio, y_test_audio = train_test_split(
    X_audio, y_audio, test_size=0.3, random_state=42, stratify=y_audio
)

print(f"Shape of X_train_audio: {X_train_audio.shape}")
print(f"Shape of X_test_audio: {X_test_audio.shape}")
print(f"Shape of y_train_audio: {y_train_audio.shape}")
print(f"Shape of y_test_audio: {y_test_audio.shape}")

**Reasoning**:
Now that the data has been split into training and testing sets, the next step is to initialize and train a Gaussian Naive Bayes model using the `X_train_audio` and `y_train_audio` datasets, as per the subtask instructions.



from sklearn.naive_bayes import GaussianNB

# 4. Initialize and train the Gaussian Naive Bayes model
model_gaussian_audio = GaussianNB()
model_gaussian_audio.fit(X_train_audio, y_train_audio)

print("Gaussian Naive Bayes model (audio features) trained successfully.")

**Reasoning**:
Now that the Gaussian Naive Bayes model has been trained, the next logical step is to evaluate its performance by making predictions on the test set and then calculating the accuracy, classification report, and confusion matrix, as outlined in the subtask instructions.



from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 5. Make predictions on the test set
y_pred_audio = model_gaussian_audio.predict(X_test_audio)

# 6. Evaluate the model's performance
print("Gaussian Naive Bayes (Audio Features) Model Performance:")
print(f"Accuracy: {accuracy_score(y_test_audio, y_pred_audio):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_audio, y_pred_audio))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_audio, y_pred_audio))

## Discretize Audio Features and Train Categorical Naive Bayes (Audio Features for Popularity)

### Subtask:
Discretize the continuous audio features from the prepared dataset into categorical bins. Then, train and evaluate a Categorical Naive Bayes model to predict song popularity, assessing its performance with similar metrics.


**Reasoning**:
First, I will prepare the continuous audio features for discretization by initializing a `KBinsDiscretizer` and applying it to the `X_audio` DataFrame. This will convert the continuous features into categorical bins, which is necessary for the `CategoricalNB` model.



from sklearn.preprocessing import KBinsDiscretizer

# 1. Identify the continuous audio features in the X_audio DataFrame that need to be discretized.
# These are the columns already in X_audio (acousticness, danceability, energy, valence, tempo, loudness, speechiness, instrumentalness).
# The features_audio list already contains these column names.

# 2. Initialize a KBinsDiscretizer from sklearn.preprocessing.
# Use the strategy='quantile' and n_bins=5 to discretize the features into 5 equal-frequency bins.
# Set encode='ordinal' to output integer bins and random_state=42 for reproducibility.
discretizer_audio = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile', subsample=None, random_state=42)

# 3. Apply the fit_transform method of the KBinsDiscretizer to the X_audio DataFrame
# to create X_audio_discretized.
X_audio_discretized_array = discretizer_audio.fit_transform(X_audio[features_audio])

# 4. Create a new DataFrame X_audio_discretized_df from the X_audio_discretized array,
# ensuring the column names indicate they are binned (e.g., 'acousticness_binned').
X_audio_discretized_df = pd.DataFrame(
    X_audio_discretized_array,
    columns=[col + '_binned' for col in features_audio]
)

print("Features DataFrame head after discretization (Audio Features):")
print(X_audio_discretized_df.head())
print(f"\nShape of X_audio_discretized_df: {X_audio_discretized_df.shape}")

**Reasoning**:
Now that the continuous audio features have been discretized into `X_audio_discretized_df`, the next step is to split this data and the target variable `y_audio` into training and testing sets, then train a Categorical Naive Bayes model and evaluate its performance with accuracy, a classification report, and a confusion matrix.



from sklearn.preprocessing import KBinsDiscretizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Identify the continuous audio features in the X_audio DataFrame that need to be discretized.
# These are the columns already in X_audio (acousticness, danceability, energy, valence, tempo, loudness, speechiness, instrumentalness).
# The features_audio list already contains these column names.

# 2. Initialize a KBinsDiscretizer from sklearn.preprocessing.
# Use the strategy='quantile' and n_bins=5 to discretize the features into 5 equal-frequency bins.
# Set encode='ordinal' to output integer bins and random_state=42 for reproducibility.
discretizer_audio = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile', subsample=None, random_state=42)

# 3. Apply the fit_transform method of the KBinsDiscretizer to the X_audio DataFrame
# to create X_audio_discretized.
X_audio_discretized_array = discretizer_audio.fit_transform(X_audio[features_audio])

# 4. Create a new DataFrame X_audio_discretized_df from the X_audio_discretized array,
# ensuring the column names indicate they are binned (e.g., 'acousticness_binned').
X_audio_discretized_df = pd.DataFrame(
    X_audio_discretized_array,
    columns=[col + '_binned' for col in features_audio]
)

print("Features DataFrame head after discretization (Audio Features):")
print(X_audio_discretized_df.head())
print(f"\nShape of X_audio_discretized_df: {X_audio_discretized_df.shape}")

# 5. Split the X_audio_discretized_df and the target variable y_audio into training and testing sets
# using train_test_split from sklearn.model_selection. Use a test_size=0.3, random_state=42, and stratify=y_audio.
X_train_audio_cat, X_test_audio_cat, y_train_audio_cat, y_test_audio_cat = train_test_split(
    X_audio_discretized_df, y_audio, test_size=0.3, random_state=42, stratify=y_audio
)

print(f"\nShape of X_train_audio_cat: {X_train_audio_cat.shape}")
print(f"Shape of X_test_audio_cat: {X_test_audio_cat.shape}")
print(f"Shape of y_train_audio_cat: {y_train_audio_cat.shape}")
print(f"Shape of y_test_audio_cat: {y_test_audio_cat.shape}")

# 6. Initialize a CategoricalNB model from sklearn.naive_bayes.
model_categorical_audio = CategoricalNB()

# 7. Train the CategoricalNB model using the training data (e.g., X_train_audio_cat, y_train_audio_cat).
model_categorical_audio.fit(X_train_audio_cat, y_train_audio_cat)

print("\nCategorical Naive Bayes model (discretized audio features) trained successfully.")

# 8. Make predictions on the test set using the trained CategoricalNB model.
y_pred_audio_cat = model_categorical_audio.predict(X_test_audio_cat)

# 9. Evaluate the model's performance by calculating and printing the accuracy score,
# a classification report, and a confusion matrix.
print("\nCategorical Naive Bayes (Discretized Audio Features) Model Performance:")
print(f"Accuracy: {accuracy_score(y_test_audio_cat, y_pred_audio_cat):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_audio_cat, y_pred_audio_cat))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_audio_cat, y_pred_audio_cat))

**Reasoning**:
The `UserWarning` about small bins indicates that some bins created by `KBinsDiscretizer` had a width too close to zero, likely due to many identical values in feature 7 (`instrumentalness`). To address this, I will decrease the number of bins from 5 to 4, as suggested by the warning, which might help in creating more distinct bins.



from sklearn.preprocessing import KBinsDiscretizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Identify the continuous audio features in the X_audio DataFrame that need to be discretized.
# These are the columns already in X_audio (acousticness, danceability, energy, valence, tempo, loudness, speechiness, instrumentalness).
# The features_audio list already contains these column names.

# 2. Initialize a KBinsDiscretizer from sklearn.preprocessing.
# Use the strategy='quantile' and n_bins=4 (decreased from 5 to address the warning).
# Set encode='ordinal' to output integer bins and random_state=42 for reproducibility.
discretizer_audio = KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='quantile', subsample=None, random_state=42)

# 3. Apply the fit_transform method of the KBinsDiscretizer to the X_audio DataFrame
# to create X_audio_discretized.
X_audio_discretized_array = discretizer_audio.fit_transform(X_audio[features_audio])

# 4. Create a new DataFrame X_audio_discretized_df from the X_audio_discretized array,
# ensuring the column names indicate they are binned (e.g., 'acousticness_binned').
X_audio_discretized_df = pd.DataFrame(
    X_audio_discretized_array,
    columns=[col + '_binned' for col in features_audio]
)

print("Features DataFrame head after discretization (Audio Features):")
print(X_audio_discretized_df.head())
print(f"\nShape of X_audio_discretized_df: {X_audio_discretized_df.shape}")

# 5. Split the X_audio_discretized_df and the target variable y_audio into training and testing sets
# using train_test_split from sklearn.model_selection. Use a test_size=0.3, random_state=42, and stratify=y_audio.
X_train_audio_cat, X_test_audio_cat, y_train_audio_cat, y_test_audio_cat = train_test_split(
    X_audio_discretized_df, y_audio, test_size=0.3, random_state=42, stratify=y_audio
)

print(f"\nShape of X_train_audio_cat: {X_train_audio_cat.shape}")
print(f"Shape of X_test_audio_cat: {X_test_audio_cat.shape}")
print(f"Shape of y_train_audio_cat: {y_train_audio_cat.shape}")
print(f"Shape of y_test_audio_cat: {y_test_audio_cat.shape}")

# 6. Initialize a CategoricalNB model from sklearn.naive_bayes.
model_categorical_audio = CategoricalNB()

# 7. Train the CategoricalNB model using the training data (e.g., X_train_audio_cat, y_train_audio_cat).
model_categorical_audio.fit(X_train_audio_cat, y_train_audio_cat)

print("\nCategorical Naive Bayes model (discretized audio features) trained successfully.")

# 8. Make predictions on the test set using the trained CategoricalNB model.
y_pred_audio_cat = model_categorical_audio.predict(X_test_audio_cat)

# 9. Evaluate the model's performance by calculating and printing the accuracy score,
# a classification report, and a confusion matrix.
print("\nCategorical Naive Bayes (Discretized Audio Features) Model Performance:")
print(f"Accuracy: {accuracy_score(y_test_audio_cat, y_pred_audio_cat):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_audio_cat, y_pred_audio_cat))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_audio_cat, y_pred_audio_cat))

**Reasoning**:
The `UserWarning` about small bins persisted even after reducing `n_bins` to 4, specifically for feature 7 (`instrumentalness`). This indicates that `instrumentalness` likely has a highly skewed distribution with many identical values (e.g., zeros), making it difficult to form distinct quantile bins. To attempt to resolve this warning by further decreasing granularity, I will try reducing `n_bins` to 3 for the `KBinsDiscretizer`.



from sklearn.preprocessing import KBinsDiscretizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Identify the continuous audio features in the X_audio DataFrame that need to be discretized.
# These are the columns already in X_audio (acousticness, danceability, energy, valence, tempo, loudness, speechiness, instrumentalness).
# The features_audio list already contains these column names.

# 2. Initialize a KBinsDiscretizer from sklearn.preprocessing.
# Use the strategy='quantile' and n_bins=3 (further decreased to address the warning).
# Set encode='ordinal' to output integer bins and random_state=42 for reproducibility.
discretizer_audio = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='quantile', subsample=None, random_state=42)

# 3. Apply the fit_transform method of the KBinsDiscretizer to the X_audio DataFrame
# to create X_audio_discretized.
X_audio_discretized_array = discretizer_audio.fit_transform(X_audio[features_audio])

# 4. Create a new DataFrame X_audio_discretized_df from the X_audio_discretized array,
# ensuring the column names indicate they are binned (e.g., 'acousticness_binned').
X_audio_discretized_df = pd.DataFrame(
    X_audio_discretized_array,
    columns=[col + '_binned' for col in features_audio]
)

print("Features DataFrame head after discretization (Audio Features):")
print(X_audio_discretized_df.head())
print(f"\nShape of X_audio_discretized_df: {X_audio_discretized_df.shape}")

# 5. Split the X_audio_discretized_df and the target variable y_audio into training and testing sets
# using train_test_split from sklearn.model_selection. Use a test_size=0.3, random_state=42, and stratify=y_audio.
X_train_audio_cat, X_test_audio_cat, y_train_audio_cat, y_test_audio_cat = train_test_split(
    X_audio_discretized_df, y_audio, test_size=0.3, random_state=42, stratify=y_audio
)

print(f"\nShape of X_train_audio_cat: {X_train_audio_cat.shape}")
print(f"Shape of X_test_audio_cat: {X_test_audio_cat.shape}")
print(f"Shape of y_train_audio_cat: {y_train_audio_cat.shape}")
print(f"Shape of y_test_audio_cat: {y_test_audio_cat.shape}")

# 6. Initialize a CategoricalNB model from sklearn.naive_bayes.
model_categorical_audio = CategoricalNB()

# 7. Train the CategoricalNB model using the training data (e.g., X_train_audio_cat, y_train_audio_cat).
model_categorical_audio.fit(X_train_audio_cat, y_train_audio_cat)

print("\nCategorical Naive Bayes model (discretized audio features) trained successfully.")

# 8. Make predictions on the test set using the trained CategoricalNB model.
y_pred_audio_cat = model_categorical_audio.predict(X_test_audio_cat)

# 9. Evaluate the model's performance by calculating and printing the accuracy score,
# a classification report, and a confusion matrix.
print("\nCategorical Naive Bayes (Discretized Audio Features) Model Performance:")
print(f"Accuracy: {accuracy_score(y_test_audio_cat, y_pred_audio_cat):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_audio_cat, y_pred_audio_cat))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_audio_cat, y_pred_audio_cat))

**Reasoning**:
The `UserWarning` about small bins persists for `instrumentalness` even after reducing `n_bins` to 3, indicating that the `quantile` strategy is not suitable for this highly skewed feature. To attempt to resolve this, I will change the `KBinsDiscretizer` strategy from `'quantile'` to `'uniform'`, which creates bins of equal width and might better handle features with many identical values.



from sklearn.preprocessing import KBinsDiscretizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Identify the continuous audio features in the X_audio DataFrame that need to be discretized.
# These are the columns already in X_audio (acousticness, danceability, energy, valence, tempo, loudness, speechiness, instrumentalness).
# The features_audio list already contains these column names.

# 2. Initialize a KBinsDiscretizer from sklearn.preprocessing.
# Use the strategy='uniform' (changed from 'quantile') and n_bins=3.
# Set encode='ordinal' to output integer bins and random_state=42 for reproducibility.
discretizer_audio = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='uniform', subsample=None, random_state=42)

# 3. Apply the fit_transform method of the KBinsDiscretizer to the X_audio DataFrame
# to create X_audio_discretized.
X_audio_discretized_array = discretizer_audio.fit_transform(X_audio[features_audio])

# 4. Create a new DataFrame X_audio_discretized_df from the X_audio_discretized array,
# ensuring the column names indicate they are binned (e.g., 'acousticness_binned').
X_audio_discretized_df = pd.DataFrame(
    X_audio_discretized_array,
    columns=[col + '_binned' for col in features_audio]
)

print("Features DataFrame head after discretization (Audio Features):")
print(X_audio_discretized_df.head())
print(f"\nShape of X_audio_discretized_df: {X_audio_discretized_df.shape}")

# 5. Split the X_audio_discretized_df and the target variable y_audio into training and testing sets
# using train_test_split from sklearn.model_selection. Use a test_size=0.3, random_state=42, and stratify=y_audio.
X_train_audio_cat, X_test_audio_cat, y_train_audio_cat, y_test_audio_cat = train_test_split(
    X_audio_discretized_df, y_audio, test_size=0.3, random_state=42, stratify=y_audio
)

print(f"\nShape of X_train_audio_cat: {X_train_audio_cat.shape}")
print(f"Shape of X_test_audio_cat: {X_test_audio_cat.shape}")
print(f"Shape of y_train_audio_cat: {y_train_audio_cat.shape}")
print(f"Shape of y_test_audio_cat: {y_test_audio_cat.shape}")

# 6. Initialize a CategoricalNB model from sklearn.naive_bayes.
model_categorical_audio = CategoricalNB()

# 7. Train the CategoricalNB model using the training data (e.g., X_train_audio_cat, y_train_audio_cat).
model_categorical_audio.fit(X_train_audio_cat, y_train_audio_cat)

print("\nCategorical Naive Bayes model (discretized audio features) trained successfully.")

# 8. Make predictions on the test set using the trained CategoricalNB model.
y_pred_audio_cat = model_categorical_audio.predict(X_test_audio_cat)

# 9. Evaluate the model's performance by calculating and printing the accuracy score,
# a classification report, and a confusion matrix.
print("\nCategorical Naive Bayes (Discretized Audio Features) Model Performance:")
print(f"Accuracy: {accuracy_score(y_test_audio_cat, y_pred_audio_cat):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_audio_cat, y_pred_audio_cat))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_audio_cat, y_pred_audio_cat))

## Prepare Lyrics Data for Popularity Classification

### Subtask:
Using the already loaded `df_lyrics` and `X_lyrics_tfidf`, create a new binary 'is_hit' target variable based on the 'track_popularity' column within the lyrics dataset. This will ensure the lyrics model also predicts popularity.


**Reasoning**:
To prepare the target variable for predicting popularity using lyrics data, I will extract the `is_hit_popularity` column from `df_lyrics`.



import pandas as pd

# 1. Define the target variable y_lyrics_popularity using the is_hit_popularity column from the df_lyrics DataFrame.
y_lyrics_popularity = df_lyrics['is_hit_popularity']

print("Value counts for y_lyrics_popularity (is_hit_popularity):")
print(y_lyrics_popularity.value_counts())
print(f"Shape of y_lyrics_popularity: {y_lyrics_popularity.shape}")

**Reasoning**:
Now that the target variable `y_lyrics_popularity` has been defined, the next step is to split the TF-IDF features (`X_lyrics_tfidf`) and this target variable into training and testing sets. This is essential for preparing the data for model training and subsequent evaluation.



from sklearn.model_selection import train_test_split

# Filter out rows from X_lyrics_tfidf that correspond to genres removed earlier
# The X_lyrics_tfidf_filtered was already created in the genre classification part.
# Need to ensure X_lyrics_tfidf and y_lyrics_popularity align after potential filtering for stratification.

# Re-align X_lyrics_tfidf with df_lyrics to ensure consistent indices
# X_lyrics_tfidf is a sparse matrix, so we convert it to a DataFrame for easier handling of indices
X_lyrics_tfidf_df = pd.DataFrame.sparse.from_spmatrix(X_lyrics_tfidf, columns=tfidf_vectorizer.get_feature_names_out())

# The target y_lyrics_popularity is based on df_lyrics. Let's ensure X matches y's index.
# Since df_lyrics was merged and potentially re-indexed, we should ensure X_lyrics_tfidf_df also reflects this.
# However, X_lyrics_tfidf was generated from df_lyrics['cleaned_lyrics'] before merging, so its shape should directly correspond to the original df_lyrics.
# The issue might be with the `single_occurrence_genres` filtering for the genre classification task.
# For popularity prediction, we should use the full X_lyrics_tfidf and y_lyrics_popularity (which is also from the full df_lyrics).

# Drop any rows where popularity_from_audio (and thus is_hit_popularity) is NaN, if any exist
# Although we filled NaN with 0, a direct drop might be safer if we want to ensure valid target values.
# However, the previous step showed 8600 zeros and 74 ones, so no NaN in y_lyrics_popularity

# 2. Split the X_lyrics_tfidf and y_lyrics_popularity into training and testing sets
# Use stratify=y_lyrics_popularity to ensure a balanced representation of 'hits' and 'non-hits'
# Given the high imbalance (8600 vs 74), stratification is crucial.
# Ensure X_lyrics_tfidf and y_lyrics_popularity have consistent indices before splitting.

# We need to make sure y_lyrics_popularity and X_lyrics_tfidf have the same number of samples and indices.
# y_lyrics_popularity came from df_lyrics, which had 8674 rows.
# X_lyrics_tfidf also has 8674 rows.

X_train_lyrics, X_test_lyrics, y_train_lyrics, y_test_lyrics = train_test_split(
    X_lyrics_tfidf,
    y_lyrics_popularity,
    test_size=0.2,
    random_state=42,
    stratify=y_lyrics_popularity
)

print(f"Shape of X_train_lyrics: {X_train_lyrics.shape}")
print(f"Shape of X_test_lyrics: {X_test_lyrics.shape}")
print(f"Shape of y_train_lyrics: {y_train_lyrics.shape}")
print(f"Shape of y_test_lyrics: {y_test_lyrics.shape}")

**Reasoning**:
Now that the data is prepared and split into training and testing sets, the next step is to initialize and train a Multinomial Naive Bayes classifier to predict song popularity based on lyrics.



from sklearn.naive_bayes import MultinomialNB

# 3. Initialize a MultinomialNB classifier
model_mnb_popularity = MultinomialNB()

# 4. Train the MultinomialNB model using the training data
model_mnb_popularity.fit(X_train_lyrics, y_train_lyrics)

print("Multinomial Naive Bayes model (lyrics for popularity) trained successfully.")

**Reasoning**:
Now that the Multinomial Naive Bayes model for lyrics-based popularity prediction has been trained, the next step is to evaluate its performance using metrics like accuracy, classification report, and confusion matrix.



from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 5. Make predictions on the test set using the trained MultinomialNB model for popularity
y_pred_mnb_popularity = model_mnb_popularity.predict(X_test_lyrics)

# 6. Evaluate the model's performance
print("Multinomial Naive Bayes (Lyrics Features for Popularity) Model Performance:")
accuracy_mnb_popularity = accuracy_score(y_test_lyrics, y_pred_mnb_popularity)
print(f"Accuracy: {accuracy_mnb_popularity:.4f}")

print("\nClassification Report:")
print(classification_report(y_test_lyrics, y_pred_mnb_popularity))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_lyrics, y_pred_mnb_popularity))

**Reasoning**:
The previous evaluation showed that the Multinomial Naive Bayes model for popularity prediction is suffering from severe class imbalance, resulting in a misleadingly high accuracy and zero precision/recall for the minority class ('hit' songs). To address this, I will apply the Synthetic Minority Over-sampling Technique (SMOTE) to the training data to balance the classes before retraining the model.



from imblearn.over_sampling import SMOTE
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Instantiate SMOTE
smote = SMOTE(random_state=42)

# Apply SMOTE to the training data
X_train_lyrics_smote, y_train_lyrics_smote = smote.fit_resample(X_train_lyrics, y_train_lyrics)

print(f"Shape of X_train_lyrics after SMOTE: {X_train_lyrics_smote.shape}")
print(f"Value counts of y_train_lyrics after SMOTE:\n{y_train_lyrics_smote.value_counts()}")

# Re-initialize and train the MultinomialNB classifier with the balanced data
model_mnb_popularity_smote = MultinomialNB()
model_mnb_popularity_smote.fit(X_train_lyrics_smote, y_train_lyrics_smote)

print("\nMultinomial Naive Bayes model (lyrics for popularity, with SMOTE) trained successfully.")

# Make predictions on the original (unbalanced) test set
y_pred_mnb_popularity_smote = model_mnb_popularity_smote.predict(X_test_lyrics)

# Evaluate the model's performance
print("\nMultinomial Naive Bayes (Lyrics Features for Popularity, with SMOTE) Model Performance:")
accuracy_mnb_popularity_smote = accuracy_score(y_test_lyrics, y_pred_mnb_popularity_smote)
print(f"Accuracy: {accuracy_mnb_popularity_smote:.4f}")

print("\nClassification Report:")
print(classification_report(y_test_lyrics, y_pred_mnb_popularity_smote))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_lyrics, y_pred_mnb_popularity_smote))

**Reasoning**:
Despite applying SMOTE, the Multinomial Naive Bayes model still failed to correctly classify any 'hit' songs, indicating persistent issues with predicting the minority class. To provide a clear visual representation of this performance, I will update the `df_accuracies` with the accuracy from the SMOTE-trained model and then visualize the confusion matrix as a heatmap.



import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Update the model accuracies DataFrame with the MNB accuracy after SMOTE
model_accuracies.append({'Model': 'Multinomial Naive Bayes (Lyrics, SMOTE)', 'Accuracy': accuracy_mnb_popularity_smote})
df_accuracies = pd.DataFrame(model_accuracies)

print("Updated Model Accuracies:")
print(df_accuracies)

# 5. Visualize the confusion matrix as a heatmap
plt.figure(figsize=(7, 6))
sns.heatmap(confusion_matrix(y_test_lyrics, y_pred_mnb_popularity_smote), annot=True, fmt='d', cmap='Reds',
            xticklabels=['Predicted 0 (Non-Hit)', 'Predicted 1 (Hit)'],
            yticklabels=['Actual 0 (Non-Hit)', 'Actual 1 (Hit)']) # Added labels
plt.title('Confusion Matrix: Multinomial Naive Bayes (Lyrics, SMOTE)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()